In [28]:
!pip install unsloth

In [29]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [30]:
max_seq_length = 2048 
dtype = None          
load_in_4bit = True   


In [31]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [32]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [33]:
dataset = load_dataset("InMedData/Cardio_v1", split="train")

In [34]:
cpt_dataset = dataset

In [35]:
def format_and_append_eos(example):

    title = str(example.get("title", ""))
    abst = str(example.get("abst", ""))

    combined_text = f"Title: {title}\nAbstract: {abst}\n"
    
    return {"text": combined_text + tokenizer.eos_token}

In [36]:
cpt_dataset = cpt_dataset.map(format_and_append_eos, remove_columns=['Unnamed: 0', 'title', 'abst'])


Map:   0%|          | 0/2600900 [00:00<?, ? examples/s]

In [37]:
cpt_dataset

Dataset({
    features: ['text'],
    num_rows: 2600900
})

In [40]:
args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    dataset_num_proc = 3, 
)


In [41]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = cpt_dataset,
    args = args,
)

Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/2600900 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=3):   0%|          | 0/2600900 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [43]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 553,036 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Step,Training Loss
1,1.571769
2,1.412266
3,1.589748
4,1.576923
5,1.323747
6,1.503926
7,1.711031
8,1.497874
9,1.730771
10,1.600816


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [44]:
adapter_name = "qwen2.5-3b-cardiology-lora"
model.save_pretrained(adapter_name)
tokenizer.save_pretrained(adapter_name)

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-cardiology-lora/tokenizer_config.json.


('qwen2.5-3b-cardiology-lora/tokenizer_config.json',
 'qwen2.5-3b-cardiology-lora/tokenizer.json')

In [45]:
!zip -r qwen2.5-3b-cardiology-lora.zip /kaggle/working/qwen2.5-3b-cardiology-lora

  adding: kaggle/working/qwen2.5-3b-cardiology-lora/ (stored 0%)
  adding: kaggle/working/qwen2.5-3b-cardiology-lora/tokenizer_config.json (deflated 89%)
  adding: kaggle/working/qwen2.5-3b-cardiology-lora/adapter_config.json (deflated 58%)
  adding: kaggle/working/qwen2.5-3b-cardiology-lora/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/qwen2.5-3b-cardiology-lora/README.md (deflated 65%)
  adding: kaggle/working/qwen2.5-3b-cardiology-lora/tokenizer.json (deflated 81%)


In [46]:
from IPython.display import FileLink

FileLink('/kaggle/working/qwen2.5-3b-cardiology-lora.zip')

/kaggle/working/qwen2.5-3b-cardiology-lora.zip

In [47]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [49]:
test_prompt = "Title: Clinical Management of Acute Myocardial Infarction\nAbstract: "
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

In [50]:
with model.disable_adapter():
    base_outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
    print(tokenizer.batch_decode(base_outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Title: Clinical Management of Acute Myocardial Infarction
Abstract: 1. Introduction 2. Epidemiology 3. Pathophysiology 4. Clinical Presentation 5. Diagnosis 6. Treatment 7. Prognosis 8. References
1. Introduction Acute myocardial infarction (AMI) is a common and serious medical condition. It is the leading cause of death in the United States. The incidence of AMI is increasing in the United States, and the number of deaths from AMI is also increasing. The incidence of AMI


In [51]:
ft_outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
print(tokenizer.batch_decode(ft_outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Title: Clinical Management of Acute Myocardial Infarction
Abstract: 1. Acute myocardial infarction (AMI) is a leading cause of death worldwide. 2. The clinical presentation of AMI is variable, and the diagnosis is often delayed. 3. The primary goal of treatment is to restore blood flow to the ischemic myocardium. 4. The treatment of AMI is guided by the TIMI (thrombolysis in myocardial infarction) classification system. 5. The TIMI classification system is based on the
